# Language models: n-grams & perplexity

_Deep Learning_

**A language model puts a probability on a sentence — n-grams estimate it from counts, and perplexity grades it.**

A language model assigns a probability to a whole sentence. "the cat sat on the mat" should score higher than "mat the on cat sat the" — the model has learned what real English looks like.

       Scoring a whole sentence at once is hopeless: there are too many possible sentences. So we factor the sentence probability into a product of next-word probabilities, one word at a time, using the chain rule of probability. Each factor is "how likely is this word, given everything before it."

---

This notebook is a practice scaffold for the **Language models: n-grams & perplexity** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Pure Python (collections.Counter, math)

### Step 1 — Build a tiny corpus and pad sentence boundaries

We start with six short sentences. Each sentence is wrapped with a start token `<s>` and an end token `</s>` so the model can learn which words tend to *begin* and *end* a sentence. We also collect the vocabulary (every distinct token, including the boundary markers), whose size `V` is needed for smoothing later.

In [ ]:
import math
from collections import Counter

# tiny training corpus (one sentence per line)
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat ran on the mat",
    "the dog ran on the log",
    "a cat sat on a mat",
    "a dog sat on a log",
]

def pad(sentence):
    # mark the start and end of every sentence
    return ["<s>"] + sentence.split() + ["</s>"]

# vocabulary, including the boundary tokens
vocab = set(["<s>", "</s>"])
for s in corpus:
    vocab.update(s.split())
V = len(vocab)

### Step 2 — Count unigrams and bigrams, define the smoothed probability

A bigram model estimates *P(current word | previous word)* from counts. We tally each word (`unigram`, the contexts) and each adjacent word pair (`bigram`). The probability uses **add-one (Laplace) smoothing**: adding 1 to the numerator and `V` to the denominator so unseen pairs get a small non-zero probability instead of zero.

In [ ]:
# count unigrams (contexts) and bigrams (word pairs)
unigram = Counter()
bigram = Counter()
for s in corpus:
    w = pad(s)
    for i in range(len(w) - 1):
        unigram[w[i]] += 1
        bigram[(w[i], w[i + 1])] += 1

def bigram_prob(prev, cur):
    # P(cur | prev) with add-one (Laplace) smoothing
    return (bigram[(prev, cur)] + 1) / (unigram[prev] + V)

### Step 3 — Score a held-out sentence and compute perplexity

Now we grade the model on a sentence it never trained on. We multiply the per-word probabilities via the chain rule, but work in **log-space** (summing logs) to avoid numerical underflow. The average negative log-probability is the cross-entropy in nats, and **perplexity = exp(cross-entropy)** — roughly the model's average number of equally-likely next-word guesses.

In [ ]:
# score a HELD-OUT sentence the model did not train on
test = "the dog sat on the mat"
w = pad(test)
log_prob = 0.0
N = 0
for i in range(len(w) - 1):
    p = bigram_prob(w[i], w[i + 1])
    log_prob += math.log(p)          # work in log-space to avoid underflow
    N += 1
    print("P(%4s | %4s) = %.4f" % (w[i + 1], w[i], p))

avg_cross_entropy = -log_prob / N    # nats per word
perplexity = math.exp(avg_cross_entropy)

print()
print("vocab size V          =", V)
print("tokens scored         =", N)
print("avg cross-entropy     = %.4f nats" % avg_cross_entropy)
print("PERPLEXITY = exp(CE)  = %.2f" % perplexity)

# ---- actual output ----
# P( the |  <s>) = 0.2941
# P( dog |  the) = 0.1579
# P( sat |  dog) = 0.2143
# P(  on |  sat) = 0.3333
# P( the |   on) = 0.2941
# P( mat |  the) = 0.1579
# P(</s> |  mat) = 0.2857
#
# vocab size V          = 11
# tokens scored         = 7
# avg cross-entropy     = 1.4330 nats
# PERPLEXITY = exp(CE)  = 4.19

## Visualize the data & results

_On a small corpus, how does held-out perplexity change as we go from a unigram to a bigram to a trigram model? Does more context always help?_

### Step 1 — Set up training and held-out corpora

To compare model orders fairly we keep two corpora: an 8-sentence training set and 2 held-out test sentences the model never sees. The `pad` helper now takes `n` so it adds `n-1` start markers — a trigram needs two `<s>` tokens of context before the first real word. The shared vocabulary is built from the training words plus boundary tokens.

In [ ]:
import math
from collections import Counter

# 8-sentence training corpus + 2 held-out test sentences
train = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat ran on the mat",
    "the dog ran on the log",
    "a cat sat on a mat",
    "a dog sat on a log",
    "the cat sat on a log",
    "the dog ran on the mat",
]
test = [
    "the cat sat on the log",
    "a dog ran on the mat",
]

def pad(sentence, n):
    # n-1 start markers, then the words, then one end marker
    return ["<s>"] * (n - 1) + sentence.split() + ["</s>"]

# shared vocabulary (training words + boundary tokens)
vocab = set(["<s>", "</s>"])
for s in train:
    vocab.update(s.split())
V = len(vocab)

### Step 2 — Build counts and a perplexity function for any order n

These two helpers generalize the bigram code to any order. `build_counts(n)` tallies every n-word run and its (n-1)-word prefix context. `perplexity(n)` scores the held-out sentences with the same add-one smoothing and returns `exp(avg cross-entropy)`. Writing them as functions of `n` lets us sweep unigram, bigram, and trigram with one call each.

In [ ]:
def build_counts(n):
    ngram = Counter()   # full n-word run
    ctx = Counter()     # its (n-1)-word prefix
    for s in train:
        w = pad(s, n)
        for i in range(len(w) - n + 1):
            gram = tuple(w[i:i + n])
            ngram[gram] += 1
            ctx[gram[:-1]] += 1
    return ngram, ctx

def perplexity(n):
    ngram, ctx = build_counts(n)
    log_prob, N = 0.0, 0
    for s in test:
        w = pad(s, n)
        for i in range(len(w) - n + 1):
            gram = tuple(w[i:i + n])
            c_full = ngram.get(gram, 0)
            c_ctx = ctx.get(gram[:-1], 0)
            p = (c_full + 1) / (c_ctx + V)   # add-one (Laplace) smoothing
            log_prob += math.log(p)
            N += 1
    return math.exp(-log_prob / N)           # exp(avg cross-entropy)

print("vocab |V| =", V)
for n in (1, 2, 3):
    print("n=%d  held-out perplexity = %.2f" % (n, perplexity(n)))

# ---- actual output ----
# vocab |V| = 11
# n=1  held-out perplexity = 9.30
# n=2  held-out perplexity = 4.17   <- best: context helps
# n=3  held-out perplexity = 4.68   <- rises: trigrams too sparse, overfits

### Step 3 — Plot perplexity versus model order

Lower perplexity is better. The bar chart shows the classic U-shape: moving from unigram to bigram helps because a little context is informative, but the trigram rises again — its contexts are too sparse on this tiny corpus, so it overfits.

In [ ]:
import matplotlib.pyplot as plt
orders = ["unigram", "bigram", "trigram"]
vals = [perplexity(1), perplexity(2), perplexity(3)]
plt.bar(orders, vals, color=["#4ea1ff", "#7ee787", "#ffb454"])
plt.ylabel("held-out perplexity (lower is better)")
plt.title("Perplexity drops then rises as n grows")
for i, v in enumerate(vals):
    plt.text(i, v, "%.2f" % v, ha="center", va="bottom")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Train a bigram model on the dog barks and the dog runs. Using raw counts (no smoothing), what is $P(\text{the dog runs})$? Treat the first word "the" as given.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count "the dog": it appears twice. Count "the": twice. So $P(\text{dog}\mid\text{the}) = 2/2 = 1.0$. — _The count ratio is how often "dog" follows "the" out of all "the" occurrences._
- Count "dog runs": once. Count "dog": twice. So $P(\text{runs}\mid\text{dog}) = 1/2 = 0.5$. — _"dog" is followed by "barks" once and "runs" once._
- Multiply the chain: $P(\text{the dog runs}) = 1.0 \times 0.5 = 0.5$. — _The chain rule makes a sentence probability the product of its next-word probabilities._

**Answer:** $P(\text{the dog runs}) = P(\text{dog}\mid\text{the}) \cdot P(\text{runs}\mid\text{dog}) = 1.0 \times 0.5 = 0.5$.

</details>

**Problem 2.** A bigram model gives a 10-word held-out sentence an average cross-entropy of $2.3$ nats per word. What is its perplexity, and what does that number mean in plain words?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the identity $\text{PP} = \exp(\text{average cross-entropy}) = \exp(2.3)$. — _Perplexity is just the exponential of the per-word cross-entropy._
- $\exp(2.3) \approx 10.0$. — _$e^{2.3} \approx 9.97$, which rounds to $10$._
- Read it as a branching factor: about 10. — _Perplexity is the model's average number of equally-likely next-word guesses._

**Answer:** Perplexity $\approx \exp(2.3) \approx 10$. The model is, on average, as uncertain about the next word as if it were guessing uniformly among 10 words at each step. Lower would be better.

</details>

**Problem 3.** You move from a bigram to a trigram model and held-out perplexity goes up, not down, even though the trigram sees more context. What is happening, and name one fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- More context per prediction should help — but trigram counts are far sparser than bigram counts. — _There are many more possible trigrams than bigrams, so most trigram contexts have very few examples._
- With tiny counts the estimates are noisy and the model fits training quirks instead of real patterns — it overfits. — _Low data per context means high variance in the estimated probabilities._
- Fix: smooth more aggressively (or back off to bigram/unigram estimates), or simply gather more training data. — _Smoothing and back-off borrow strength from lower orders; more data fills in the sparse trigram contexts._

**Answer:** Data sparsity: trigram contexts have too few examples, so the model overfits and held-out perplexity rises. Fix it with stronger smoothing or back-off (blend in bigram/unigram estimates), or train on more text.

</details>